# Sequential (Recursive) Forecasting — Greek IDA2/IDA3 Intraday Prices

**Approach:** Train a 1-step-ahead model (15 min), then recursively predict step-by-step up to 12 steps (3h).
Each prediction is fed back as a feature for the next step.

**Models:** LSTM, MLP, XGBoost, Linear Regression — same architecture families and Optuna search as the direct multi-horizon notebook.

**Evaluation:** Steps 4, 8, 12 (= 1h, 2h, 3h) reported for fair comparison with direct models.


In [ ]:
!pip install optuna xgboost -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import itertools, random, os, time, warnings

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 1. Data Loading and Feature Engineering

Identical pipeline to the direct multi-horizon notebook, plus IDA_target added as an explicit feature for the recursive feedback loop.


In [ ]:
FILENAME = '/content/Final2026_with_ENTSOE.csv'

try:
    df = pd.read_csv(FILENAME, sep=',')
    if len(df.columns) == 1: df = pd.read_csv(FILENAME, sep=';')
except: df = pd.read_csv(FILENAME)

# Rename columns to internal names used throughout the notebook
df = df.rename(columns={
    'Day time': 'DELIVERY_MTU',
    'DAM price': 'DAM_MCP',
    'IDA1': 'MCP',
    'IDA2': 'MCP_IDA2',
    'IDA3': 'MCP_IDA3',
    'BM price': 'BM_IMBALANCE_PRICE',
    'BMmDAM price': 'BMmDAMMCP',
})

df['DELIVERY_MTU'] = pd.to_datetime(df['DELIVERY_MTU'])
df = df.sort_values('DELIVERY_MTU').reset_index(drop=True)

# Time features
df['hour']      = df['DELIVERY_MTU'].dt.hour
df['minute']    = df['DELIVERY_MTU'].dt.minute / 60.0
df['dayofweek'] = df['DELIVERY_MTU'].dt.dayofweek
df['ida3_available'] = df['MCP_IDA3'].notna().astype(float)

# Fill missing — use global ffill/bfill for multi-day gaps
df['MCP_IDA3']  = df['MCP_IDA3'].fillna(0.0)
df['MCP_IDA2']  = df['MCP_IDA2'].ffill().bfill().fillna(0.0)
df['MCP']       = df['MCP'].ffill().bfill().fillna(df['MCP_IDA2'])
df['DAM_MCP']   = df['DAM_MCP'].ffill().bfill()
df['BMmDAMMCP'] = df['BMmDAMMCP'].ffill().bfill().fillna(0.0)
df['BM_IMBALANCE_PRICE'] = df['BM_IMBALANCE_PRICE'].ffill().bfill().fillna(df['BM_IMBALANCE_PRICE'].median())
df['IDA_target'] = np.where(df['ida3_available'] == 1, df['MCP_IDA3'], df['MCP_IDA2'])
df['ISP2_RES']  = df['ISP2_RES'].ffill().bfill().fillna(df['ISP1_RES'])
df['ISP2_Load'] = df['ISP2_Load'].ffill().bfill().fillna(df['ISP1_Load'])

# ── ISP revision features ──
df['DA_RES'] = df['Wind_DA_forecast'] + df['Solar_DA_forecast']
is_ida3 = df['hour'] >= 12

# Latest available ISP (ISP2 for IDA2, ISP3 for IDA3 — no leakage)
df['latest_RES']  = np.where(is_ida3, df['ISP3_RES'], df['ISP2_RES'])
df['latest_Load'] = np.where(is_ida3, df['ISP3_Load'], df['ISP2_Load'])

# Revision features (delta from DA)
df['latest_RES_revision']  = df['latest_RES'] - df['DA_RES']
df['latest_Load_revision'] = df['latest_Load'] - df['SystemLoad_DA_forecast']
df['latest_net_load']      = df['latest_Load'] - df['latest_RES']
df['DA_net_load']          = df['SystemLoad_DA_forecast'] - df['DA_RES']
df['latest_net_load_revision'] = df['latest_net_load'] - df['DA_net_load']

# ISP2->ISP3 sequential revision (only for IDA3 rows)
df['RES_revision_ISP2_to_ISP3']  = np.where(is_ida3, df['ISP3_RES'] - df['ISP2_RES'], 0.0)
df['Load_revision_ISP2_to_ISP3'] = np.where(is_ida3, df['ISP3_Load'] - df['ISP2_Load'], 0.0)

# ── BM and momentum features ──
df['bm_roll_mean_4']  = df['BM_IMBALANCE_PRICE'].rolling(4, min_periods=1).mean()
df['bm_roll_std_4']   = df['BM_IMBALANCE_PRICE'].rolling(4, min_periods=1).std().fillna(0)
df['bm_roll_mean_12'] = df['BM_IMBALANCE_PRICE'].rolling(12, min_periods=1).mean()
df['ida_momentum']  = df['MCP_IDA2'].diff(4).fillna(0)
df['mcp_momentum']  = df['MCP'].diff(4).fillna(0)
df['bm_momentum']   = df['BM_IMBALANCE_PRICE'].diff(4).fillna(0)
df['bm_ida_spread']  = df['BM_IMBALANCE_PRICE'] - df['MCP_IDA2']
df['ida_roll_std_4'] = df['MCP_IDA2'].rolling(4, min_periods=1).std().fillna(0)

# ── 1-step target for sequential model ──
df['IDA_target_1step'] = df['IDA_target'].shift(-1)

# Also create multi-step targets for evaluation reference
for h_steps, h_name in [(4, '1h'), (8, '2h'), (12, '3h')]:
    df[f'IDA_target_{h_name}'] = df['IDA_target'].shift(-h_steps)
    df[f'ida3_at_{h_name}'] = (df['hour'].shift(-h_steps) >= 12).astype(float)

df = df.dropna(subset=['IDA_target_1step', 'IDA_target_1h', 'IDA_target_2h', 'IDA_target_3h']).reset_index(drop=True)

# Date filter — same as direct notebook
df = df[df['DELIVERY_MTU'] >= '2025-10-02'].copy().reset_index(drop=True)
df = df[df['DELIVERY_MTU'] <= '2026-01-18 23:45:00'].copy().reset_index(drop=True)

# ── Feature set: original 30 + IDA_target for recursive feedback ──
FEATURES_BASE = [
    # Prices
    'MCP', 'MCP_IDA2', 'DAM_MCP',
    # BM signal
    'BM_IMBALANCE_PRICE', 'bm_roll_mean_4', 'bm_roll_std_4', 'bm_roll_mean_12', 'bm_ida_spread',
    # Momentum
    'ida_momentum', 'mcp_momentum', 'bm_momentum',
    # Volatility
    'ida_roll_std_4',
    # DA context
    'SystemLoad_DA_forecast', 'Wind_DA_forecast', 'Solar_DA_forecast',
    # Auction flags
    'ida3_available', 'ida3_at_1h', 'ida3_at_2h', 'ida3_at_3h',
    # Time
    'hour', 'minute', 'dayofweek',
    # ISP raw (latest available, respecting info timeline)
    'latest_RES', 'latest_Load',
    # ISP revisions (delta from DA)
    'latest_RES_revision', 'latest_Load_revision', 'latest_net_load_revision', 'latest_net_load',
    # ISP2->ISP3 sequential change
    'RES_revision_ISP2_to_ISP3', 'Load_revision_ISP2_to_ISP3',
]

# Add IDA_target as feature for the recursive loop
FEATURES = FEATURES_BASE + ['IDA_target']
IDA_TARGET_IDX = len(FEATURES) - 1  # Index of IDA_target in feature vector

TARGET = 'IDA_target_1step'  # 1-step-ahead target
HORIZON = 1  # Single-step output
BATCH = 64
TRAIN_RATIO = 0.8
split_idx = int(len(df) * TRAIN_RATIO)
N_FEAT = len(FEATURES)

# Check for any remaining NaNs in features
nan_check = df[FEATURES].isna().sum()
if nan_check.sum() > 0:
    print("WARNING: NaN in features:")
    print(nan_check[nan_check > 0])
    df[FEATURES] = df[FEATURES].ffill().bfill().fillna(0)
    print("Fixed with ffill/bfill/fillna(0)")

print(f"Dataset: {df.shape}, Features: {N_FEAT} (30 original + IDA_target)")
print(f"Train: {split_idx}, Test: {len(df)-split_idx}")
print(f"Days: {df['DELIVERY_MTU'].dt.date.nunique()}")
print(f"Period: {df['DELIVERY_MTU'].min()} to {df['DELIVERY_MTU'].max()}")



## 2. Evaluation Metrics

Same metrics as the direct notebook: MAE, RMSE, R², FSI.


In [ ]:
def compute_metrics(y_true, y_pred, y_naive):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mae_naive = mean_absolute_error(y_true, y_naive)
    fsi  = 1 - mae / mae_naive  # Forecast Skill Index
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'FSI': fsi}

print("Metrics ready: MAE, RMSE, R2, FSI")



## 3. Data Scaling and Datasets

MinMaxScaler fitted on train only. HORIZON=1 (single output) for sequential approach.


In [ ]:
# Scale features
scaler_X = MinMaxScaler()
data_X = df[FEATURES].values.astype(np.float32)
scaler_X.fit(data_X[:split_idx])
data_X_sc = scaler_X.transform(data_X)

# Scale target (1-step)
scaler_y = MinMaxScaler()
all_tgt = df[TARGET].values.astype(np.float32).reshape(-1, 1)
scaler_y.fit(all_tgt[:split_idx])
tgt_sc = scaler_y.transform(all_tgt).flatten()

# PyTorch datasets — HORIZON=1 for sequential
class LSTMDataset(Dataset):
    def __init__(self, X, y, lb):
        self.X, self.y, self.lb = X, y, lb
    def __len__(self): return len(self.X) - self.lb
    def __getitem__(self, i):
        return (torch.tensor(self.X[i:i+self.lb], dtype=torch.float32),
                torch.tensor(self.y[i+self.lb], dtype=torch.float32))

class MLPDataset(Dataset):
    def __init__(self, X, y, lb):
        self.X, self.y, self.lb = X, y, lb
    def __len__(self): return len(self.X) - self.lb
    def __getitem__(self, i):
        return (torch.tensor(self.X[i:i+self.lb].flatten(), dtype=torch.float32),
                torch.tensor(self.y[i+self.lb], dtype=torch.float32))

print(f"Scaling ready. Target shape: {tgt_sc.shape}")
print(f"IDA_target feature index: {IDA_TARGET_IDX}")



## 4. Model Definitions

Same architectures as direct notebook but with HORIZON=1 output.


In [ ]:
class SeqLSTM(nn.Module):
    """LSTM for 1-step-ahead prediction (same architecture as PointLSTM with horizon=1)."""
    def __init__(self, n_feat, hidden, layers, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.fc = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden // 2, 1))
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1]).squeeze(-1)  # (batch,)

class SeqMLP(nn.Module):
    """MLP for 1-step-ahead prediction (same architecture as PointMLP with horizon=1)."""
    def __init__(self, input_dim, hidden, n_layers, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for _ in range(n_layers):
            layers += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(dropout)]
            prev = hidden
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)  # (batch,)

def train_nn(model, ld_tr, ld_te, epochs, lr, label="", verbose=True):
    model = model.to(device)
    crit = nn.L1Loss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=8, factor=0.5)
    best_test = float('inf'); best_state = None
    for ep in range(1, epochs+1):
        model.train(); tl = 0
        for X, y in ld_tr:
            X, y = X.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(X), y); loss.backward(); opt.step()
            tl += loss.item() * X.size(0)
        tl /= len(ld_tr.dataset)
        model.eval(); vl = 0
        with torch.no_grad():
            for X, y in ld_te:
                X, y = X.to(device), y.to(device)
                vl += crit(model(X), y).item() * X.size(0)
        vl /= len(ld_te.dataset)
        sched.step(vl)
        if vl < best_test:
            best_test = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if verbose and (ep % 30 == 0 or ep == 1):
            print(f"  [{label}] {ep:3d}/{epochs} | train: {tl:.6f} | test: {vl:.6f}")
    model.load_state_dict(best_state)
    if verbose: print(f"  [{label}] Best: {best_test:.6f}")
    return best_test

print("Sequential models defined (LSTM, MLP - horizon=1)")



## 5. Optuna — MLP Hyperparameter Search (1-step)

Same search space as direct notebook (lookback: [24,48,96], hidden: [64,128,256,512], layers: 2-5).


In [ ]:
SEARCH_EPOCHS = 60

def mlp_objective(trial):
    lb = trial.suggest_categorical('lookback', [24, 48, 96])
    hidden = trial.suggest_categorical('hidden', [64, 128, 256, 512])
    n_layers = trial.suggest_int('n_layers', 2, 5)
    dropout = trial.suggest_float('dropout', 0.1, 0.4, step=0.1)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)

    tr = MLPDataset(data_X_sc[:split_idx], tgt_sc[:split_idx], lb)
    te = MLPDataset(data_X_sc[split_idx-lb:], tgt_sc[split_idx-lb:], lb)
    ld_tr = DataLoader(tr, batch_size=BATCH, shuffle=True)
    ld_te = DataLoader(te, batch_size=BATCH, shuffle=False)

    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = SeqMLP(lb * N_FEAT, hidden, n_layers, dropout).to(device)
    crit = nn.L1Loss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=7, factor=0.5)
    best_val = float('inf')
    for ep in range(1, SEARCH_EPOCHS + 1):
        model.train()
        for X, y in ld_tr:
            X, y = X.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(X), y); loss.backward(); opt.step()
        model.eval(); vl = 0
        with torch.no_grad():
            for X, y in ld_te:
                X, y = X.to(device), y.to(device)
                vl += crit(model(X), y).item() * X.size(0)
        vl /= len(ld_te.dataset); sched.step(vl)
        if vl < best_val: best_val = vl
        trial.report(vl, ep)
        if trial.should_prune(): raise optuna.TrialPruned()
    return best_val

print("Running MLP Optuna (50 trials, 1-step target)...")
mlp_study = optuna.create_study(direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=15),
    sampler=optuna.samplers.TPESampler(seed=SEED))
mlp_study.optimize(mlp_objective, n_trials=50, show_progress_bar=True)
bp_mlp = mlp_study.best_trial.params
print(f"MLP best: {bp_mlp}, val_loss={mlp_study.best_trial.value:.6f}")



## 6. Optuna — LSTM Hyperparameter Search (1-step)


In [ ]:
def lstm_objective(trial):
    lb = trial.suggest_categorical('lookback', [24, 48])
    hidden = trial.suggest_categorical('hidden', [64, 128])
    layers = trial.suggest_int('layers', 1, 2)
    lr = trial.suggest_float('lr', 5e-4, 3e-3, log=True)

    tr = LSTMDataset(data_X_sc[:split_idx], tgt_sc[:split_idx], lb)
    te = LSTMDataset(data_X_sc[split_idx-lb:], tgt_sc[split_idx-lb:], lb)
    ld_tr = DataLoader(tr, batch_size=BATCH, shuffle=True)
    ld_te = DataLoader(te, batch_size=BATCH, shuffle=False)

    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = SeqLSTM(N_FEAT, hidden, layers).to(device)
    crit = nn.L1Loss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=7, factor=0.5)
    best_val = float('inf')
    for ep in range(1, SEARCH_EPOCHS + 1):
        model.train()
        for X, y in ld_tr:
            X, y = X.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(X), y); loss.backward(); opt.step()
        model.eval(); vl = 0
        with torch.no_grad():
            for X, y in ld_te:
                X, y = X.to(device), y.to(device)
                vl += crit(model(X), y).item() * X.size(0)
        vl /= len(ld_te.dataset); sched.step(vl)
        if vl < best_val: best_val = vl
        trial.report(vl, ep)
        if trial.should_prune(): raise optuna.TrialPruned()
    return best_val

print("Running LSTM Optuna (30 trials, 1-step target)...")
lstm_study = optuna.create_study(direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=15),
    sampler=optuna.samplers.TPESampler(seed=SEED))
lstm_study.optimize(lstm_objective, n_trials=30, show_progress_bar=True)
bp_lstm = lstm_study.best_trial.params
print(f"LSTM best: {bp_lstm}, val_loss={lstm_study.best_trial.value:.6f}")



## 7. XGBoost Grid Search (1-step)


In [ ]:
def build_xgb_data(lookback):
    data = df[FEATURES].values.astype(np.float32)
    targets = df[TARGET].values.astype(np.float32)
    n = len(data) - lookback
    X = np.zeros((n, lookback * N_FEAT), dtype=np.float32)
    y = np.zeros(n, dtype=np.float32)
    for i in range(n):
        X[i] = data[i:i+lookback].flatten()
        y[i] = targets[i + lookback]
    s = split_idx - lookback
    return X[:s], X[s:], y[:s], y[s:]

# XGBoost grid search (same grid as direct notebook)
print('XGBoost Grid Search (1-step target)...')
best_mae = float('inf')
bp_xgb = {'lookback': 24, 'max_depth': 4, 'n_estimators': 200,
           'learning_rate': 0.05, 'subsample': 0.8}
for lb in [24, 48]:
    for depth in [4, 6]:
        for n_est in [200, 400]:
            for lr in [0.05, 0.1]:
                X_tr, X_te, y_tr, y_te = build_xgb_data(lb)
                m = xgb.XGBRegressor(max_depth=depth, n_estimators=n_est,
                    learning_rate=lr, subsample=0.8, colsample_bytree=0.8,
                    objective='reg:absoluteerror', tree_method='hist',
                    n_jobs=-1, verbosity=0)
                m.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
                mae = mean_absolute_error(y_te, m.predict(X_te))
                if mae < best_mae:
                    best_mae = mae
                    bp_xgb = {'lookback': lb, 'max_depth': depth,
                              'n_estimators': n_est, 'learning_rate': lr,
                              'subsample': 0.8}
                    print(f'  New best: lb={lb} depth={depth} trees={n_est} lr={lr} MAE={mae:.2f}')
print(f'\nXGBoost best: {bp_xgb}')



## 8. Train Final 1-Step Models


In [ ]:
EPOCHS = 120

# ── MLP ──
print("=" * 60)
print(f"MLP (1-step): {bp_mlp}")
lb = bp_mlp['lookback']
tr = MLPDataset(data_X_sc[:split_idx], tgt_sc[:split_idx], lb)
te = MLPDataset(data_X_sc[split_idx-lb:], tgt_sc[split_idx-lb:], lb)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model_mlp = SeqMLP(lb*N_FEAT, bp_mlp['hidden'], bp_mlp['n_layers'], bp_mlp['dropout'])
train_nn(model_mlp, DataLoader(tr, batch_size=BATCH, shuffle=True),
         DataLoader(te, batch_size=BATCH, shuffle=False), EPOCHS, bp_mlp['lr'], "MLP")

# ── LSTM ──
print("=" * 60)
print(f"LSTM (1-step): {bp_lstm}")
lb = bp_lstm['lookback']
tr = LSTMDataset(data_X_sc[:split_idx], tgt_sc[:split_idx], lb)
te = LSTMDataset(data_X_sc[split_idx-lb:], tgt_sc[split_idx-lb:], lb)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model_lstm = SeqLSTM(N_FEAT, bp_lstm['hidden'], bp_lstm['layers'])
train_nn(model_lstm, DataLoader(tr, batch_size=BATCH, shuffle=True),
         DataLoader(te, batch_size=BATCH, shuffle=False), EPOCHS, bp_lstm['lr'], "LSTM")

# ── XGBoost ──
print("=" * 60)
print(f"XGBoost (1-step): {bp_xgb}")
lb_xgb = bp_xgb['lookback']
X_tr_xgb, X_te_xgb, y_tr_xgb, y_te_xgb = build_xgb_data(lb_xgb)
model_xgb = xgb.XGBRegressor(max_depth=bp_xgb['max_depth'], n_estimators=bp_xgb['n_estimators'],
    learning_rate=bp_xgb['learning_rate'], subsample=bp_xgb['subsample'],
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    objective='reg:absoluteerror', tree_method='hist', n_jobs=-1, verbosity=0)
model_xgb.fit(X_tr_xgb, y_tr_xgb, eval_set=[(X_te_xgb, y_te_xgb)], verbose=False)
print(f"  XGBoost 1-step MAE: {mean_absolute_error(y_te_xgb, model_xgb.predict(X_te_xgb)):.2f}")

# ── Linear Regression ──
print("=" * 60)
print("Linear Regression (1-step)")
lb_lr = 48
X_tr_lr, X_te_lr, y_tr_lr, y_te_lr = build_xgb_data(lb_lr)
model_lr = LinearRegression()
model_lr.fit(X_tr_lr, y_tr_lr)
print(f"  Linear 1-step MAE: {mean_absolute_error(y_te_lr, model_lr.predict(X_te_lr)):.2f}")

print("\nAll 4 one-step models trained.")



## 9. Recursive (Sequential) Inference — 12 Steps

The core of the sequential approach:
1. Start with the real lookback window at time t
2. Predict IDA_target at t+1 (scaled space for NN, raw for XGB/LR)
3. Construct synthetic feature row: copy last real row, replace IDA_target with prediction
4. Append synthetic row to window, slide forward by 1
5. Repeat 12 times to get predictions at t+1 through t+12

Exogenous features (BM, ISP, DA, time) are **frozen at last known values** — only IDA_target updates recursively.


In [ ]:
MAX_STEPS = 12  # Predict up to 12 steps (3h) ahead

def recursive_predict_nn(model, data_sc, start_idx, lookback, n_steps, model_type='lstm'):
    """
    Recursively predict n_steps ahead using a trained 1-step NN model.

    Protocol:
    - Start with real lookback window ending at start_idx
    - Predict 1 step ahead
    - Construct synthetic next row: copy last row, replace IDA_target with prediction
    - Slide window forward, repeat
    - Exogenous features frozen at last known values
    """
    model.eval()
    # Copy the lookback window
    window = data_sc[start_idx - lookback + 1 : start_idx + 1].copy()  # (lookback, n_feat)
    predictions = []

    with torch.no_grad():
        for step in range(n_steps):
            if model_type == 'lstm':
                x = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)
            else:  # mlp
                x = torch.tensor(window.flatten(), dtype=torch.float32).unsqueeze(0).to(device)

            pred_sc = model(x).cpu().item()  # scaled prediction
            pred_real = scaler_y.inverse_transform([[pred_sc]])[0, 0]
            predictions.append(pred_real)

            # Construct next row: copy last row, update IDA_target only
            next_row = window[-1].copy()
            next_row[IDA_TARGET_IDX] = pred_sc  # Feed prediction back (scaled space)

            # Slide window: drop oldest, append synthetic row
            window = np.vstack([window[1:], next_row.reshape(1, -1)])

    return np.array(predictions)


def recursive_predict_flat(model, data_raw, start_idx, lookback, n_steps):
    """
    Recursively predict n_steps ahead using XGBoost or Linear Regression.
    Same protocol but in raw (unscaled) feature space.
    """
    window = data_raw[start_idx - lookback + 1 : start_idx + 1].copy()
    predictions = []

    for step in range(n_steps):
        x = window.flatten().reshape(1, -1)
        pred = model.predict(x)[0]
        predictions.append(pred)

        # Next row: copy last, update IDA_target
        next_row = window[-1].copy()
        next_row[IDA_TARGET_IDX] = pred  # Raw space
        window = np.vstack([window[1:], next_row.reshape(1, -1)])

    return np.array(predictions)

print("Recursive prediction functions defined.")
print(f"  IDA_target feature index: {IDA_TARGET_IDX}")
print(f"  Max recursive steps: {MAX_STEPS} ({MAX_STEPS * 15} min = {MAX_STEPS * 15 / 60:.0f}h)")



## 10. Run Recursive Inference on Full Test Set

For each test sample, recursively predict 12 steps. Store predictions at all 12 steps.


In [ ]:
# Test range: must have lookback available AND 12 future steps for evaluation
test_start = split_idx
test_end = len(df) - MAX_STEPS  # Need 12 future actuals
n_test = test_end - test_start

print(f"Test samples: {n_test} (indices {test_start} to {test_end-1})")
print(f"Lookbacks - MLP: {bp_mlp['lookback']}, LSTM: {bp_lstm['lookback']}, XGB: {lb_xgb}, LR: {lb_lr}")

# Raw feature data for XGBoost/LR
data_raw = df[FEATURES].values.astype(np.float32)

# Storage: (n_test, 12) for each model
all_recursive_preds = {}

# ── MLP recursive ──
print("\nMLP recursive inference...")
preds_mlp_rec = np.zeros((n_test, MAX_STEPS))
lb = bp_mlp['lookback']
model_mlp.eval()
for i in range(n_test):
    idx = test_start + i
    if idx >= lb:
        preds_mlp_rec[i] = recursive_predict_nn(
            model_mlp, data_X_sc, idx, lb, MAX_STEPS, 'mlp')
    if (i+1) % 500 == 0: print(f"  {i+1}/{n_test}")
all_recursive_preds['MLP'] = preds_mlp_rec
print(f"  Done. Shape: {preds_mlp_rec.shape}")

# ── LSTM recursive ──
print("\nLSTM recursive inference...")
preds_lstm_rec = np.zeros((n_test, MAX_STEPS))
lb = bp_lstm['lookback']
model_lstm.eval()
for i in range(n_test):
    idx = test_start + i
    if idx >= lb:
        preds_lstm_rec[i] = recursive_predict_nn(
            model_lstm, data_X_sc, idx, lb, MAX_STEPS, 'lstm')
    if (i+1) % 500 == 0: print(f"  {i+1}/{n_test}")
all_recursive_preds['LSTM'] = preds_lstm_rec
print(f"  Done. Shape: {preds_lstm_rec.shape}")

# ── XGBoost recursive ──
print("\nXGBoost recursive inference...")
preds_xgb_rec = np.zeros((n_test, MAX_STEPS))
for i in range(n_test):
    idx = test_start + i
    if idx >= lb_xgb:
        preds_xgb_rec[i] = recursive_predict_flat(
            model_xgb, data_raw, idx, lb_xgb, MAX_STEPS)
    if (i+1) % 500 == 0: print(f"  {i+1}/{n_test}")
all_recursive_preds['XGBoost'] = preds_xgb_rec
print(f"  Done. Shape: {preds_xgb_rec.shape}")

# ── Linear Regression recursive ──
print("\nLinear Regression recursive inference...")
preds_lr_rec = np.zeros((n_test, MAX_STEPS))
for i in range(n_test):
    idx = test_start + i
    if idx >= lb_lr:
        preds_lr_rec[i] = recursive_predict_flat(
            model_lr, data_raw, idx, lb_lr, MAX_STEPS)
    if (i+1) % 500 == 0: print(f"  {i+1}/{n_test}")
all_recursive_preds['Linear'] = preds_lr_rec
print(f"  Done. Shape: {preds_lr_rec.shape}")

print(f"\nAll recursive predictions complete: {n_test} origins x {MAX_STEPS} steps")



## 11. Collect Actuals and Naive Baseline


In [ ]:
# Actuals at each step offset from origin
actuals_all_steps = np.zeros((n_test, MAX_STEPS))
ida_target_vals = df['IDA_target'].values.astype(np.float32)

for i in range(n_test):
    idx = test_start + i
    for s in range(MAX_STEPS):
        actuals_all_steps[i, s] = ida_target_vals[idx + s + 1]

# Naive baseline: last known IDA_target (persistence forecast)
naive_vals = ida_target_vals[test_start:test_start + n_test]

print(f"Actuals shape: {actuals_all_steps.shape}")
print(f"Naive baseline: last known IDA_target (persistence)")



## 12. Results — Recursive Model at Every Step

MAE, RMSE, R², and FSI for each of the 12 recursive steps. Steps 4, 8, 12 (= 1h, 2h, 3h) highlighted for comparison with the direct notebook.


In [ ]:
# Full results at every step
print("  SEQUENTIAL (RECURSIVE) MODEL COMPARISON — ALL 12 STEPS")
print()

for name, preds in all_recursive_preds.items():
    print(f"  === {name} ===")
    print(f"  {'Step':<8} {'Horizon':>8} {'MAE':>8} {'RMSE':>8} {'R2':>7} {'FSI':>7}")
    print(f"  {'-'*55}")
    for s in range(MAX_STEPS):
        horizon_str = f"{(s+1)*15}min"
        m = compute_metrics(
            actuals_all_steps[:, s],
            preds[:, s],
            naive_vals
        )
        marker = " <--" if (s+1) in [4, 8, 12] else ""
        h_label = {4: " (1h)", 8: " (2h)", 12: " (3h)"}.get(s+1, "")
        print(f"  t+{s+1:<5} {horizon_str:>8} {m['MAE']:>8.2f} {m['RMSE']:>8.2f} {m['R2']:>7.3f} {m['FSI']*100:>+6.1f}%{h_label}{marker}")
    print()

# Naive reference
print(f"  === Naive (persistence) ===")
print(f"  {'Step':<8} {'Horizon':>8} {'MAE':>8} {'RMSE':>8}")
print(f"  {'-'*40}")
for s in range(MAX_STEPS):
    horizon_str = f"{(s+1)*15}min"
    mae_n = mean_absolute_error(actuals_all_steps[:, s], naive_vals)
    rmse_n = np.sqrt(mean_squared_error(actuals_all_steps[:, s], naive_vals))
    marker = " <--" if (s+1) in [4, 8, 12] else ""
    h_label = {4: " (1h)", 8: " (2h)", 12: " (3h)"}.get(s+1, "")
    print(f"  t+{s+1:<5} {horizon_str:>8} {mae_n:>8.2f} {rmse_n:>8.2f}{h_label}{marker}")



## 13. MAE at All 12 Recursive Steps

Shows error accumulation as the recursion deepens.


In [ ]:
print("  MAE at each recursive step (15-min intervals)")
print(f"  {'Step':<8} {'Minutes':>7}", end="")
for name in all_recursive_preds:
    print(f" {name:>10}", end="")
print(f" {'Naive':>10}")
print(f"  {'-'*80}")

mae_by_step = {name: [] for name in all_recursive_preds}
mae_naive_by_step = []

for s in range(MAX_STEPS):
    print(f"  t+{s+1:<5} {(s+1)*15:>5}min", end="")
    for name, preds in all_recursive_preds.items():
        mae = mean_absolute_error(actuals_all_steps[:, s], preds[:, s])
        mae_by_step[name].append(mae)
        print(f" {mae:>10.2f}", end="")
    mae_n = mean_absolute_error(actuals_all_steps[:, s], naive_vals)
    mae_naive_by_step.append(mae_n)
    print(f" {mae_n:>10.2f}")



## 14. Error Accumulation Plot

Visualize how MAE grows with each recursive step.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = {'MLP': 'darkorange', 'LSTM': 'green', 'XGBoost': 'purple', 'Linear': 'brown'}
steps = np.arange(1, MAX_STEPS + 1)
minutes = steps * 15

for name, maes in mae_by_step.items():
    ax.plot(minutes, maes, 'o-', color=colors[name], label=name, linewidth=2, markersize=5)

ax.plot(minutes, mae_naive_by_step, 's--', color='gray', label='Naive', linewidth=2, markersize=5)

# Mark evaluation points (1h, 2h, 3h)
for step_min in [60, 120, 180]:
    ax.axvline(step_min, color='lightgray', linestyle=':', alpha=0.7)
    ax.text(step_min+2, ax.get_ylim()[1]*0.95 if ax.get_ylim()[1] > 0 else 30, f'{step_min//60}h', fontsize=10, color='gray')

ax.set_xlabel('Forecast Horizon (minutes)')
ax.set_ylabel('MAE (EUR/MWh)')
ax.set_title('Sequential (Recursive) Forecast - Error Accumulation by Step')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(minutes)
ax.set_xticklabels([f'{m}' for m in minutes])
plt.tight_layout()
plt.show()



## 15. Prediction Plots — 7 Test Days at Key Horizons


In [ ]:
N_PLOT = 96 * 7  # 7 days

fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)
colors = {'MLP': 'darkorange', 'LSTM': 'green', 'XGBoost': 'purple', 'Linear': 'brown'}
eval_list = [('1h (step 4)', 3), ('2h (step 8)', 7), ('3h (step 12)', 11)]

for ax, (h_name, step_idx) in zip(axes, eval_list):
    ax.plot(actuals_all_steps[:N_PLOT, step_idx], 'k-', alpha=0.6, label='Actual', linewidth=0.8)
    for name, preds in all_recursive_preds.items():
        ax.plot(preds[:N_PLOT, step_idx], color=colors[name], alpha=0.7, label=name, linewidth=0.8)
    ax.set_ylabel('EUR/MWh')
    ax.set_title(f'Sequential Forecast - {h_name}')
    ax.legend(fontsize=8, ncol=5, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Test sample index')
plt.tight_layout()
plt.show()



## 16. Rolling MAE (1h horizon)


In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))
w = 96  # 1-day rolling window
step_idx = 3  # 1h = step 4
colors = {'MLP': 'darkorange', 'LSTM': 'green', 'XGBoost': 'purple', 'Linear': 'brown'}
for name, preds in all_recursive_preds.items():
    err = np.abs(actuals_all_steps[:, step_idx] - preds[:, step_idx])
    ax.plot(pd.Series(err).rolling(w).mean(), color=colors[name], label=name, linewidth=1)
# Naive
err_naive = np.abs(actuals_all_steps[:, step_idx] - naive_vals)
ax.plot(pd.Series(err_naive).rolling(w).mean(), color='gray', label='Naive', linewidth=1, linestyle='--')

ax.set_xlabel('Test sample'); ax.set_ylabel('Rolling MAE (EUR/MWh)')
ax.set_title('Sequential Forecast - Rolling MAE at t+1h (96-sample window)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()



## 17. Error Distributions


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
eval_list = [('1h (step 4)', 3), ('2h (step 8)', 7), ('3h (step 12)', 11)]
colors = {'MLP': 'darkorange', 'LSTM': 'green', 'XGBoost': 'purple', 'Linear': 'brown'}

for ax, (h_name, step_idx) in zip(axes, eval_list):
    for name, preds in all_recursive_preds.items():
        errors = actuals_all_steps[:, step_idx] - preds[:, step_idx]
        ax.hist(errors, bins=60, alpha=0.4, label=name, color=colors[name])
    ax.set_title(f'Error Distribution - {h_name}')
    ax.set_xlabel('Error (EUR/MWh)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()



## 18. IDA2 vs IDA3 Breakdown


In [ ]:
test_hours = df['hour'].values[test_start:test_start+n_test]
ida2_mask = test_hours < 12
ida3_mask = test_hours >= 12

print("  IDA2 (hours < 12) vs IDA3 (hours >= 12) - MAE at 1h, 2h, 3h")
print(f"  {'Model':<16} {'IDA2-1h':>8} {'IDA2-2h':>8} {'IDA2-3h':>8} | {'IDA3-1h':>8} {'IDA3-2h':>8} {'IDA3-3h':>8}")
print(f"  {'-'*85}")

for name, preds in all_recursive_preds.items():
    print(f"  {name:<16}", end="")
    for step_idx in [3, 7, 11]:
        mae2 = mean_absolute_error(actuals_all_steps[ida2_mask, step_idx], preds[ida2_mask, step_idx])
        print(f" {mae2:>8.2f}", end="")
    print(" |", end="")
    for step_idx in [3, 7, 11]:
        mae3 = mean_absolute_error(actuals_all_steps[ida3_mask, step_idx], preds[ida3_mask, step_idx])
        print(f" {mae3:>8.2f}", end="")
    print()

# Naive
print(f"  {'Naive':<16}", end="")
for step_idx in [3, 7, 11]:
    mae2 = mean_absolute_error(actuals_all_steps[ida2_mask, step_idx], naive_vals[ida2_mask])
    print(f" {mae2:>8.2f}", end="")
print(" |", end="")
for step_idx in [3, 7, 11]:
    mae3 = mean_absolute_error(actuals_all_steps[ida3_mask, step_idx], naive_vals[ida3_mask])
    print(f" {mae3:>8.2f}", end="")
print()



## 19. Export Predictions


In [ ]:
ts = df['DELIVERY_MTU'].values[test_start:test_start+n_test]
out = pd.DataFrame({'timestamp': ts})
out['hour'] = pd.to_datetime(ts).hour

# Add predictions at all 12 steps for each model
for name, preds in all_recursive_preds.items():
    for s in range(MAX_STEPS):
        out[f'{name}_step{s+1}'] = preds[:, s]

# Add actuals at all 12 steps
for s in range(MAX_STEPS):
    out[f'actual_step{s+1}'] = actuals_all_steps[:, s]

out['naive'] = naive_vals

out.to_csv('sequential_predictions.csv', index=False)
print(f"Exported: {out.shape}")
print(out.head())



## 20. Summary

| Model | Approach | HP Search | Feedback | Horizons |
|---|---|---|---|---|
| **MLP** | Recursive 1-step | Optuna 50 trials | IDA_target fed back | 12 steps (15min each) |
| **LSTM** | Recursive 1-step | Optuna 30 trials | IDA_target fed back | 12 steps (15min each) |
| **XGBoost** | Recursive 1-step | 16-config grid | IDA_target fed back | 12 steps (15min each) |
| **Linear** | Recursive 1-step | Fixed lb=48 | IDA_target fed back | 12 steps (15min each) |

**Recursive protocol:**
- Model trained on 1-step-ahead target (15 min)
- At inference: predict t+1, update IDA_target feature with prediction, shift window, repeat
- Exogenous features (BM, ISP, DA, time) frozen at last known values
- Evaluated at steps 4, 8, 12 (= 1h, 2h, 3h) for comparison with direct models

**Key question for thesis:** Does error accumulation in the recursive approach outweigh the potential benefit of step-by-step refinement? The error accumulation plot (Section 14) answers this directly.
